In [1]:
def convert_to_minutes(data):
    parts = data.split(":")

    day = int(parts[0])
    hour = int(parts[1])
    minute = int(parts[2])

    if not (1 <= day <= 31):
        return -1
    if not (1 <= hour <= 23):
        return -1
    if not (1 <= minute <= 59):
        return -1

    total_minutes = (day - 1) * 24 * 60 + hour * 60 + minute

    return total_minutes

In [2]:
def get_call_data(file_name):
    result_list = []

    file = open(file_name, "r")
    file.readline()

    for line in file:
        line = line.strip()

        if line == "":
            continue
        if line.startswith("#"):
            continue
        parts = line.split(";")

        caller_id = int(parts[0])
        receiver_id = int(parts[1])
        caller_cell_id = int(parts[2])
        receiver_cell_id = int(parts[3])

        if caller_id == receiver_id:
            continue

        start_total = convert_to_minutes(parts[4])
        end_total = convert_to_minutes(parts[5])

        if start_total == -1:
            continue
        if end_total == -1:
            continue

        if start_total > end_total:
            continue

        duration = end_total - start_total + 1

        if duration >= 600:
            continue

        valid_call = (caller_id, receiver_id, caller_cell_id, receiver_cell_id, duration)
        result_list.append(valid_call)

    file.close()
    return result_list

In [3]:
def calculate_bills(call_data):
    total_minutes = {}  # Dictionary to accumulate minutes

    for call in call_data:
        caller = call[0]
        duration = call[4]

        if caller not in total_minutes:
            total_minutes[caller] = duration
        else:
            total_minutes[caller] += duration

    bills = {}
    for customer in total_minutes:
        minutes = total_minutes[customer]

        # Applying the 20-minute bonus
        payable_minutes = max(0, minutes - 20)

        bills[customer] = payable_minutes * 0.10

    return bills

In [4]:
def calculate_congested_cells(call_data):
    traffic_areas = {}

    for area in call_data:
        zone = area[2]

        if zone not in traffic_areas:
            traffic_areas[zone] = 1
        else:
            traffic_areas[zone] += 1

    total_calls = sum(traffic_areas.values())
    cell_count = len(traffic_areas)
    if cell_count == 0:
        return []

    average = total_calls / cell_count

    congested_cells = []
    for cell_id in traffic_areas:
        call_count = traffic_areas[cell_id]

        if traffic_areas[cell_id] > average:
            congested_cells.append(cell_id)

    return congested_cells

In [7]:
# --- Execution ---

# 1. Transform the file into a clean list
data = get_call_data("telefonate.csv")

# 2. Use that list for billing
bills = calculate_bills(data)

# 3. Use the same list for cell analysis
critical_cells = calculate_congested_cells(data)

print("Calculated bills:", bills)
print("Cells above average:", critical_cells)

Calculated bills: {25: 3.1, 35: 4.1000000000000005, 10: 7.4, 39: 3.0, 12: 1.6, 41: 5.0, 34: 6.1000000000000005, 30: 6.6000000000000005, 23: 5.300000000000001, 9: 2.7, 42: 4.5, 27: 3.5, 4: 1.3, 43: 2.4000000000000004, 6: 4.0, 48: 4.0, 49: 5.4, 24: 3.3000000000000003, 3: 5.2, 19: 3.1, 20: 5.6000000000000005, 17: 3.5, 8: 5.800000000000001, 32: 6.800000000000001, 0: 4.3, 11: 6.2, 45: 4.0, 37: 0.0, 31: 2.8000000000000003, 2: 3.2, 1: 4.1000000000000005, 16: 1.6, 14: 6.7, 7: 4.2, 15: 6.1000000000000005, 13: 0.0, 44: 3.1, 33: 1.3, 18: 2.1, 5: 5.4, 36: 2.8000000000000003, 46: 5.300000000000001, 38: 2.0, 26: 0.0, 47: 2.2, 40: 1.2000000000000002, 29: 2.9000000000000004, 21: 1.6, 28: 4.6000000000000005, 22: 2.8000000000000003}
Cells above average: [5, 13, 7, 1, 3, 8, 14, 4, 6, 2, 19, 15, 17]
